# Rokoko Mocap Baker Pipeline

Source files: `Assets/Scripts/Rokoko/RokokoMocapBaker.cs` (Inspector/UI layer),
`Assets/Scripts/Rokoko/RokokoJsonlBaker.cs` (bake engine), `Assets/Rokoko/Scripts/Mono/Inputs/Actor.cs`
(Rokoko package, retargeting logic reused unchanged by the baker).

Input: a `rokoko_skeleton.jsonl` recording, one JSON object per line, each line holding a
`studio_timestamp` and an `actors` array. Output: a Humanoid `AnimationClip` asset (muscle
curves + root motion curves), droppable into an Animator Controller like any other
Humanoid clip.

Every stage below runs the exact formulas and constants from the source files, in plain
Python, against **real values from `documentation/rokoko_skeleton.jsonl`** -- the recording
captured for actor `"Daniel"` in this project -- and the real character T-pose stored on
this project's own `PeteCharacter` Actor component (`Assets/_JENII/Scenes/JN_with_robot.unity`),
so each stage's input and output is visible as real numbers, not stand-ins. Only Stage 5's
muscle/root values (where Unity's own `HumanPoseHandler` runs, not reproducible outside the
engine -- see that stage) and Stage 7's keyframe *values* (as opposed to their *times*, which
are real) stay as clearly-labeled illustrative placeholders.

In [16]:
import json
import math

def q_mul(a, b):
    """Unity Quaternion multiplication, (x, y, z, w) tuples."""
    x1, y1, z1, w1 = a
    x2, y2, z2, w2 = b
    w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
    x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
    y = w1 * y2 + y1 * w2 + z1 * x2 - x1 * z2
    z = w1 * z2 + z1 * w2 + x1 * y2 - y1 * x2
    return (x, y, z, w)

def q_mul3(a, b, c):
    return q_mul(q_mul(a, b), c)

def q_conjugate(q):
    x, y, z, w = q
    return (-x, -y, -z, w)

def q_inverse(q):
    x, y, z, w = q
    norm_sq = x * x + y * y + z * z + w * w
    cx, cy, cz, cw = q_conjugate(q)
    return (cx / norm_sq, cy / norm_sq, cz / norm_sq, cw / norm_sq)

def q_normalize(q):
    x, y, z, w = q
    n = math.sqrt(x * x + y * y + z * z + w * w)
    return (x / n, y / n, z / n, w / n)

def cross(a, b):
    return (
        a[1] * b[2] - a[2] * b[1],
        a[2] * b[0] - a[0] * b[2],
        a[0] * b[1] - a[1] * b[0],
    )

def rotate_vector(q, v):
    """Rotates vector v by quaternion q (Unity's `q * v`)."""
    qn = q_normalize(q)
    vq = (v[0], v[1], v[2], 0.0)
    rx, ry, rz, _ = q_mul3(qn, vq, q_conjugate(qn))
    return (rx, ry, rz)

def fmt(q, nd=4):
    return tuple(round(c, nd) for c in q)

## Loading the real recording

This notebook lives in `documentation/`, right next to the recording it uses:
`documentation/rokoko_skeleton.jsonl`. Every stage below that touches recorded data reads
from this file -- nothing past this cell is hand-typed.

Each line carries more than `RawLine` (`RokokoJsonlBaker.cs:33-38`) declares --
`frame_idx`, `t_recv`, `version`, `fps` are all real fields in the file that `RawLine` never
declares a field for. `JsonUtility.FromJson` silently ignores JSON fields with no matching
C# field, so only `studio_timestamp` and `actors` ever reach the baker; the rest is the
recorder's own bookkeeping.

In [17]:
RECORDING_PATH = "rokoko_skeleton.jsonl"

with open(RECORDING_PATH, encoding="utf-8") as f:
    recording = [json.loads(line) for line in f if line.strip()]

actor_names = sorted({a["name"] for line in recording for a in line["actors"]})
timestamps = [line["studio_timestamp"] for line in recording]

print(f"{len(recording)} lines")
print(f"actor(s): {actor_names}")
print(f"studio_timestamp range: {timestamps[0]:.3f} .. {timestamps[-1]:.3f} s "
      f"({timestamps[-1] - timestamps[0]:.1f} s of motion)")
print(f"top-level keys per line: {sorted(recording[0].keys())}  <- RawLine only reads 2 of these")


1114 lines
actor(s): ['Daniel']
studio_timestamp range: 0.099 .. 37.703 s (37.6 s of motion)
top-level keys per line: ['actors', 'fps', 'frame_idx', 'studio_timestamp', 't_recv', 'version']  <- RawLine only reads 2 of these


## Stage 0a — T-pose capture

`Actor.CalculateTPose()` (`Actor.cs:98-105`) runs when the Inspector's "Assign T-Pose Now"
button is clicked, with the character posed in T-pose in the Scene view at that instant.
`InitializeCharacterTPose()` (`Actor.cs:116-128`) walks every `HumanBodyBones` value and
records `boneTransform.rotation` — the bone's *current world rotation, right now* — into
`characterTPose`. Nothing here derives from the mocap data; it is a live snapshot of
whatever pose the character's Transforms happen to be in at the moment of the click.

The values below are **not** invented -- they're the real, already-captured
`characterTPose` for the character this recording was made on: GameObject `PeteCharacter`
(`profileName: "Daniel"`, matching this recording's actor name), read straight out of its
`Actor` component's serialized `characterTPose.keys` / `characterTPose.values` in
`Assets/_JENII/Scenes/JN_with_robot.unity`. Only a 7-bone subset is shown here (matching the
`SmartsuitTPose` excerpt in Stage 0b below), out of the ~51 bones actually stored there.

In [24]:
# Real values -- Actor.characterTPose for PeteCharacter ("Daniel"), read from
# Assets/_JENII/Scenes/JN_with_robot.unity (HumanBodyBones index -> quaternion, this
# project's actual rest pose, not an example).
character_t_pose = {
    "Hips":          (0.0000046788955, 0.000015752348, -0.00049349066, 0.9999999),
    "Spine":         (-0.038748957, 1.2732926e-10, 1.1504339e-11, 0.999249),
    "Chest":         (-0.038748946, 1.2551027e-10, 1.1504353e-11, 0.999249),
    "LeftUpperArm":  (0.48209566, -0.48209655, 0.51728475, 0.51728445),
    "RightUpperArm": (-0.4818959, -0.4818919, 0.51747465, -0.51747143),
    "LeftUpperLeg":  (0.0027398933, -0.000901467, 0.9999946, 0.0016107162),
    "RightUpperLeg": (-0.0027470216, -0.0019867346, 0.99999297, -0.0016065014),
}

for bone, q in character_t_pose.items():
    print(f"{bone:<15} {fmt(q)}")


Hips            (0.0, 0.0, -0.0005, 1.0)
Spine           (-0.0387, 0.0, 0.0, 0.9992)
Chest           (-0.0387, 0.0, 0.0, 0.9992)
LeftUpperArm    (0.4821, -0.4821, 0.5173, 0.5173)
RightUpperArm   (-0.4819, -0.4819, 0.5175, -0.5175)
LeftUpperLeg    (0.0027, -0.0009, 1.0, 0.0016)
RightUpperLeg   (-0.0027, -0.002, 1.0, -0.0016)


## Stage 0b — rotation offsets

`InitializeBoneOffsets()` (`Actor.cs:133-137`) calls `CalculateRotationOffsets()`
(`Actor.cs:323-334`), which for every bone present in `characterTPose` computes:

```
offsets[bone] = Inverse(SmartsuitTPose[bone]) * characterTPose[bone]
```

`SmartsuitTPose` (`Actor.cs:339-395`) is a fixed dictionary hardcoded in the Rokoko
package — the Smartsuit's own reference T-pose rotation per bone, in the suit's world
convention. It is the same for every character, real or example; only `characterTPose`
changes. A subset of its real values (verbatim from `Actor.cs`):

In [29]:
smartsuit_t_pose = {
    "Hips":          (0.0, 0.0, 0.0, 1.0),
    "Spine":         (0.0, 0.0, 1.0, 0.0),
    "Chest":         (0.0, 0.0, 1.0, 0.0),
    "LeftUpperArm":  (-0.5, -0.5, 0.5, -0.5),
    "RightUpperArm": (0.5, -0.5, 0.5, 0.5),
    "LeftUpperLeg":  (0.0, 0.707, 0.0, 0.707),
    "RightUpperLeg": (0.0, -0.707, 0.0, 0.707),
}

offsets = {}
for bone in smartsuit_t_pose:
    offsets[bone] = q_mul(q_inverse(smartsuit_t_pose[bone]), character_t_pose[bone])
    print(f"{bone:<15} offset = {fmt(offsets[bone])}")

Hips            offset = (0.0, 0.0, -0.0005, 1.0)
Spine           offset = (0.0, 0.0387, -0.9992, 0.0)
Chest           offset = (0.0, 0.0387, -0.9992, 0.0)
LeftUpperArm    offset = (0.0352, 0.0, -0.9994, 0.0)
RightUpperArm   offset = (0.0356, 0.0, 0.9994, -0.0)
LeftUpperLeg    offset = (-0.7053, -0.0018, 0.7091, 0.0005)
RightUpperLeg   offset = (0.7053, -0.0025, 0.7092, 0.0003)


Unlike the earlier version of this notebook (which used a made-up character
whose rest pose was defined identically to the suit's, so every offset came out as
identity), `PeteCharacter`'s real rest pose doesn't exactly match `SmartsuitTPose` on every
bone. The offsets printed above are the real, non-identity per-bone corrections
`Actor.UpdateBone` applies on *every single frame* of this character's bake -- computed
once here, then reused unchanged for the rest of this walkthrough.

## Stage 1 — reading one JSONL line

Each line of `rokoko_skeleton.jsonl` deserializes into `RawLine` (`RokokoJsonlBaker.cs:33-38`):
`studio_timestamp` plus an `actors` array of `RawActorEntry` (`RokokoJsonlBaker.cs:25-31`):
`name`, `hip_height`, and `bones` (a `BodyFrame`, `JsonLiveSerializerV3.cs:83-161` — one
`ActorJointFrame { position, rotation }` per named joint). Below is the real first line of
the loaded recording, with only 2 of its real joints shown (the exact count is printed
below -- this particular capture has no finger/face tracking) so the printed output stays
readable.

In [25]:
FRAME_INDEX = 0
raw_line = recording[FRAME_INDEX]
all_joint_names = list(raw_line["actors"][0]["bones"].keys())

preview = {
    "studio_timestamp": raw_line["studio_timestamp"],
    "actors": [{
        "name": a["name"],
        "hip_height": a["hip_height"],
        "bones": {k: a["bones"][k] for k in ("hip", "leftUpperArm")},
    } for a in raw_line["actors"]],
}
print(json.dumps(preview, indent=2))
print()
print(f"(the real 'bones' dict actually has {len(all_joint_names)} joints; only 2 shown above)")

print()
print(f"All {len(all_joint_names)} joint names in this recording:")
print(all_joint_names)


{
  "studio_timestamp": 0.098842,
  "actors": [
    {
      "name": "Daniel",
      "hip_height": 0.9346893,
      "bones": {
        "hip": {
          "position": {
            "x": -0.04342729,
            "y": 0.9200311,
            "z": 0.0199824423
          },
          "rotation": {
            "x": -0.0395867676,
            "y": 0.0199850425,
            "z": 0.00560137257,
            "w": -0.9990006
          }
        },
        "leftUpperArm": {
          "position": {
            "x": -0.270062327,
            "y": 1.395759,
            "z": -0.00721970759
          },
          "rotation": {
            "x": -0.0540950671,
            "y": 0.8115236,
            "z": -0.04615966,
            "w": 0.579976
          }
        }
      }
    }
  ]
}

(the real 'bones' dict actually has 23 joints; only 2 shown above)

All 23 joint names in this recording:
['hip', 'spine', 'chest', 'neck', 'head', 'leftShoulder', 'leftUpperArm', 'leftLowerArm', 'leftHand', 'rightShoulder', '

## Stage 2 — actor selection

`BakeToClip` (`RokokoJsonlBaker.cs:146-156`) scans `raw.actors` for an entry whose `name`
matches `actor.profileName`, falling back to `actors[0]` if none matches — a mocap take
recorded with multiple performers only bakes the one this character is assigned to.

This recording's actor is `"Daniel"`, which is also `PeteCharacter`'s real `Actor.profileName`
in the scene (Stage 0a) -- an exact match, so this lookup succeeds without needing the
fallback.

In [26]:
def select_actor_entry(raw, profile_name):
    for a in raw["actors"]:
        if a["name"] == profile_name:
            return a
    return raw["actors"][0]

src_actor = select_actor_entry(raw_line, profile_name="Daniel")
preview_src = {**src_actor, "bones": dict(list(src_actor["bones"].items())[:2])}
print(json.dumps(preview_src, indent=2))
print(f"\n(again, only 2 of {len(src_actor['bones'])} real joints shown)")


{
  "name": "Daniel",
  "hip_height": 0.9346893,
  "bones": {
    "hip": {
      "position": {
        "x": -0.04342729,
        "y": 0.9200311,
        "z": 0.0199824423
      },
      "rotation": {
        "x": -0.0395867676,
        "y": 0.0199850425,
        "z": 0.00560137257,
        "w": -0.9990006
      }
    },
    "spine": {
      "position": {
        "x": -0.0410319,
        "y": 1.00721741,
        "z": -0.012431778
      },
      "rotation": {
        "x": 0.0215838645,
        "y": 0.0277129319,
        "z": -0.9993343,
        "w": 0.009859751
      }
    }
  }
}

(again, only 2 of 23 real joints shown)


## Stage 3 — building the `ActorFrame`

`RokokoJsonlBaker.cs:158-164` repackages the raw entry into Rokoko's own `ActorFrame`
struct: `name`, `meta.hasBody = true` (unconditionally — this baker only ever bakes body
data, not gloves/face), `dimensions.hipHeight = hip_height`, `body = bones`.

In [ ]:
actor_frame = {
    "name": src_actor["name"],
    "meta": {"hasBody": True},
    "dimensions": {"hipHeight": src_actor["hip_height"]},
    "body": src_actor["bones"],  # full real per-frame joint dict, not the 2-joint preview above
}
print(f"name={actor_frame['name']!r}, hipHeight={actor_frame['dimensions']['hipHeight']}, "
      f"body has {len(actor_frame['body'])} real joints")


name='Daniel', hipHeight=0.9346893, body has 23 real joints


## Stage 4 — `Actor.UpdateActor` → `UpdateSkeleton` → `UpdateBone`

`UpdateActor` (`Actor.cs:162-177`) sets `profileName = actorFrame.name`, then — since
`hasBody` is true — calls `UpdateSkeleton` (`Actor.cs:249-269`), which walks every
`HumanBodyBones` value, pulls the matching `ActorJointFrame` out of `body` by name, and
calls `UpdateBone` (`Actor.cs:274-316`) for each one.

Only `HumanBodyBones.Hips` gets `shouldUpdatePosition = true` — every other bone is
rotation-only, since a Humanoid rig derives limb position from parent rotations via
forward kinematics; the hips are the one bone that needs an explicit world position.

`UpdateBone`'s rotation branch, for this project's default `rotationSpace = Offset`
(`Actor.cs:312-315`):

```
boneTransform.rotation = Hips.parent.rotation * worldRotation * offsets[bone]
```

`worldRotation` is this frame's raw rotation for the bone, straight out of the JSONL, in
the Smartsuit's own convention. `offsets[bone]` (Stage 0b) re-expresses that rotation in
this specific character's rest-pose convention. `Hips.parent.rotation` re-anchors the
result to wherever this character's root sits in the scene. This one line is the actual
per-bone retargeting happening on every frame of every bake.

In [28]:
def update_bone_offset_mode(hips_parent_rotation, world_rotation, offset):
    return q_mul3(hips_parent_rotation, world_rotation, offset)

# Identity here because this stage is isolating the per-bone retargeting math itself --
# the character root's actual scene rotation is a separate, unrelated Transform value.
hips_parent_rotation = (0.0, 0.0, 0.0, 1.0)

# Real value: actor_frame's own "leftUpperArm" rotation, straight from this frame of the
# real recording (Stage 1/3), not hand-picked.
r = actor_frame["body"]["leftUpperArm"]["rotation"]
left_upper_arm_raw = (r["x"], r["y"], r["z"], r["w"])
print("Raw leftUpperArm rotation from the recording:", fmt(left_upper_arm_raw))

retargeted_left_upper_arm = update_bone_offset_mode(
    hips_parent_rotation,
    left_upper_arm_raw,
    offsets["LeftUpperArm"],
)
print("LeftUpperArm final local rotation:", fmt(retargeted_left_upper_arm))


Raw leftUpperArm rotation from the recording: (-0.0541, 0.8115, -0.0462, 0.58)
LeftUpperArm final local rotation: (-0.7906, -0.0557, -0.6082, -0.0442)


## Stage 4b — hip position

`UpdateBone`'s position branch (`Actor.cs:287-298`), for this project's real
`positionSpace = Self` (`Actor.positionSpace: 1` in the scene, i.e. `Space.Self`):

```
boneTransform.position = parent.rotation * worldPosition + parent.position
```

and, only if `adjustHipHeightBasedOnStudioActor` is enabled -- this project's real Actor has
it **off** (`adjustHipHeightBasedOnStudioActor: 0` in the scene) -- `UpdateBone` would first
apply (`Actor.cs:262-264`):

```
worldPosition.y -= (actorFrame.dimensions.hipHeight - characterHipHeight)
```

which re-levels a mocap performer's hip height difference against this specific
character's own leg length before the position conversion above runs.

In [ ]:
def self_space_position(parent_rotation, parent_position, world_position):
    rx, ry, rz = rotate_vector(parent_rotation, world_position)
    px, py, pz = parent_position
    return (rx + px, ry + py, rz + pz)

# Real value: this frame's actual recorded hip position (Stage 1/3), not hand-picked.
p = actor_frame["body"]["hip"]["position"]
hip_world_position_raw = (p["x"], p["y"], p["z"])
print("Raw hip position from the recording:", tuple(round(c, 4) for c in hip_world_position_raw))

hips_parent_position = (0.0, 0.0, 0.0)  # character root assumed at scene origin (see Stage 4)

character_hip_height = 0.88   # this character's own resting hip height (a per-character
                               # constant only known from measuring it in Unity -- not part
                               # of the recording, so still illustrative)
actor_hip_height = actor_frame["dimensions"]["hipHeight"]  # real, from Stage 3

adjust_hip_height = False  # this project's real Actor.adjustHipHeightBasedOnStudioActor
if adjust_hip_height:
    hx, hy, hz = hip_world_position_raw
    hip_world_position_raw = (hx, hy - (actor_hip_height - character_hip_height), hz)

final_hip_position = self_space_position(hips_parent_rotation, hips_parent_position, hip_world_position_raw)
print("Hips final local position:", tuple(round(c, 4) for c in final_hip_position))


## Stage 5 — sampling the posed rig

After `UpdateActor` finishes mutating every bone Transform for this frame,
`RokokoJsonlBaker.cs:173` calls `poseHandler.GetHumanPose(ref pose)`. `HumanPoseHandler`
is Unity's own Mecanim component; it reads the now-posed Transform hierarchy and converts
it into `HumanPose` — roughly 95 normalized `muscles[]` values (one per Humanoid degree
of freedom, in the `[-1, 1]` range defined by this specific avatar's configured muscle
limits) plus `bodyPosition`/`bodyRotation` for the root. This conversion runs inside the
Unity engine against this avatar's imported muscle-limit data and is not something to
reproduce outside Unity; everything on either side of it is plain data transformation, as
shown in every other stage here.

## Stage 6 — timestamp to keyframe time

`RokokoJsonlBaker.cs:168-171`, run once per JSONL line, in this exact order:

```
dt = 1/30                         if lastTs < 0
     max(studio_timestamp - lastTs, 1/240)   otherwise
if applied > 0: time += dt
lastTs = studio_timestamp
```

The first applied frame always lands at `time = 0` regardless of its own `dt` (that `dt`
only affects the *next* frame's advance); the `1/240` floor guards against a near-zero or
negative timestamp delta producing a degenerate or reversed keyframe.

In [ ]:
def accumulate_times(timestamps):
    last_ts = -1.0
    time = 0.0
    applied = 0
    result = []
    for ts in timestamps:
        dt = (1 / 30) if last_ts < 0 else max(ts - last_ts, 1 / 240)
        if applied > 0:
            time += dt
        last_ts = ts
        applied += 1
        result.append(round(time, 5))
    return result

# Real timestamps: the recording's own first 8 studio_timestamp values (Stage "Loading the
# real recording" above), not hand-picked.
real_timestamps = timestamps[:8]
real_times = accumulate_times(real_timestamps)
print("studio_timestamp:", real_timestamps)
print("keyframe time:   ", real_times)


## Stage 7 — writing the `AnimationClip`

For every applied frame, `RokokoJsonlBaker.cs:175-184` pushes one `Keyframe(time, value)`
per channel: `HumanTrait.MuscleCount` muscle channels (named by Unity's own
`HumanTrait.MuscleName[m]`, e.g. `"Spine Front-Back"`, `"Left Arm Twist In-Out"`) plus
three `RootT.x/y/z` and four `RootQ.x/y/z/w` channels for root motion.

`RokokoJsonlBaker.cs:203-213` then calls, once per channel:

```
clip.SetCurve("", typeof(Animator), channelName, new AnimationCurve(keyframes))
```

Targeting `typeof(Animator)` with these specific channel names — rather than recording
raw Transform position/rotation curves per bone, the way a `GameObjectRecorder` would —
is what makes the resulting `AnimationClip` a genuine Humanoid muscle clip: retargetable
to any other Humanoid avatar, and usable in an Animator Controller alongside other
Humanoid clips.

In [ ]:
# Times below are real (Stage 6's accumulated times for this recording's first 3 frames).
# Values are illustrative placeholders -- Stage 5 already covered why real muscle/root
# values only exist inside Unity's own HumanPoseHandler, not reproducible here.
t0, t1, t2 = real_times[:3]

muscle_keyframes_example = {
    "Spine Front-Back": [(t0, 0.02), (t1, 0.021), (t2, 0.019)],
    "Left Arm Twist In-Out": [(t0, -0.10), (t1, -0.11), (t2, -0.09)],
}
root_t_keyframes_example = {
    "RootT.x": [(t0, 0.01), (t1, 0.012)],
    "RootT.y": [(t0, 0.92), (t1, 0.921)],
    "RootT.z": [(t0, -0.02), (t1, -0.019)],
}
root_q_keyframes_example = {
    "RootQ.x": [(t0, 0.0), (t1, 0.001)],
    "RootQ.y": [(t0, 0.05), (t1, 0.052)],
    "RootQ.z": [(t0, 0.0), (t1, 0.0)],
    "RootQ.w": [(t0, 0.999), (t1, 0.999)],
}

for channel, keys in {**muscle_keyframes_example, **root_t_keyframes_example, **root_q_keyframes_example}.items():
    print(f"{channel:<24} {keys}")


## Rig safety around the whole loop

`RokokoJsonlBaker.cs:119-126` snapshots every Transform's `localPosition`/`localRotation`
under `actor.animator.transform` before the frame loop starts (`GetComponentsInChildren
<Transform>`), because `UpdateActor` mutates the live scene rig frame by frame to produce
each pose to sample. `RokokoJsonlBaker.cs:189-198` restores every snapshot in a `finally`
block — on normal completion or on an exception mid-loop — so baking a clip never leaves
the character's Transforms displaced in the scene.

`RokokoJsonlBaker.cs:96-97` also invokes `Actor.InitializeAnimatorHumanBones` and
`Actor.InitializeBoneOffsets` through reflection before the loop starts. Both are normally
called from `Actor.Awake()` (`Actor.cs:76-77`), which only runs in Play Mode; baking runs
entirely in the Editor without entering Play Mode, so without this call `offsets` (Stage
0b) and the cached bone-transform dictionary would still be empty at bake time.

## Inspector layer — `RokokoMocapBaker` / `RokokoMocapBakerEditor`

`RokokoMocapBaker` (`RokokoMocapBaker.cs:20-27`) itself holds only two fields:
`jsonlPath` and `outputFolder`. Every action lives in its `CustomEditor`:

- `EnsureActorSetUp` (`RokokoMocapBaker.cs:126-152`), run on every Inspector repaint: adds
  an `Actor` component if the character doesn't have one yet, wires `actor.animator`, and
  calls `CalculateTPose()` exactly once — only while `characterTPose` is still empty, so a
  T-pose already captured is never silently overwritten by whatever pose the character
  happens to be in during a later repaint.
- The "Assign T-Pose Now" / "Recalculate T-Pose" button (`RokokoMocapBaker.cs:68-76`) calls
  `actor.CalculateTPose()` on demand, then checks `actor.isValidTpose` — `Actor.cs:204-228`
  rejects the capture unless the hand-to-hand direction is within `0.99` dot product of
  world right and the spine-to-chest direction is within `0.99` dot product of world up.
- The drop area / Browse button (`RokokoMocapBaker.cs:154-179`, `81-89`) sets `jsonlPath`.
- `RokokoJsonlBaker.Validate` (`RokokoJsonlBaker.cs:44-53`) gates the "Bake Animation Clip"
  button: an `Actor` must exist, its `Animator` must be assigned and Humanoid, and
  `characterTPose` must be non-empty.
- `Bake` (`RokokoMocapBaker.cs:181-199`) normalizes `outputFolder` to a forward-slash,
  in-`Assets`-rooted path, creates it if missing, builds a unique output filename as
  `{characterName}_{jsonlBaseName}.anim` via `AssetDatabase.GenerateUniqueAssetPath`, and
  calls `RokokoJsonlBaker.RunBake`, which wraps `BakeToClip` (everything in Stages 0–7
  above) with a progress bar and a completion dialog reporting the frame count.